# CNN Detect

## Load Dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

ZIP_PATH = "/content/drive/MyDrive/Image Processing Project/openfake_subset.zip"
SPLIT_CSV = "/content/drive/MyDrive/Image Processing Project/model_split_metadata.csv"

assert os.path.exists(ZIP_PATH), f"Missing zip: {ZIP_PATH}"
assert os.path.exists(SPLIT_CSV), f"Missing split CSV: {SPLIT_CSV}"

print("Found dataset zip ✅")
print("Found split CSV ✅")

Mounted at /content/drive
Found dataset zip ✅
Found split CSV ✅


In [2]:
import zipfile
import shutil
import os

EXTRACT_DIR = "/content/dataset"

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)

os.makedirs(EXTRACT_DIR, exist_ok=True)

print("Extracting dataset...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

print("Done ✅")
print(os.listdir(EXTRACT_DIR))

Extracting dataset...
Done ✅
['images', 'metadata.csv']


In [3]:
import pandas as pd
import os

METADATA_PATH = "/content/dataset/metadata.csv"

df = pd.read_csv(METADATA_PATH)

df["image_path"] = df["image_path"].str.replace(
    "/content/openfake_subset/images",
    "/content/dataset/images",
    regex=False
)

print(df.head())
print("\nDataset shape:", df.shape)
print("\nLabels:")
print(df["label"].value_counts())

sample_path = df["image_path"].iloc[0]
print("\nSample path:", sample_path)
print("File exists:", os.path.exists(sample_path))

                      image_path  \
0  /content/dataset/images/0.jpg   
1  /content/dataset/images/1.jpg   
2  /content/dataset/images/2.jpg   
3  /content/dataset/images/3.jpg   
4  /content/dataset/images/4.jpg   

                                              prompt label model  
0  A photograph of a person wearing a white ribbe...  real  real  
1  A promotional poster for Bill Moyers' Faith & ...  real  real  
2  A promotional poster showcasing a cozy living ...  real  real  
3  A photograph of Emily Blunt at the 33rd London...  real  real  
4  A collage of three photographs showcasing mode...  real  real  

Dataset shape: (8989, 4)

Labels:
label
real                3000
stable_diffusion    3000
midjourney          2989
Name: count, dtype: int64

Sample path: /content/dataset/images/0.jpg
File exists: True


In [4]:
import os
import shutil
import pandas as pd
from tqdm.auto import tqdm

split_df = pd.read_csv(SPLIT_CSV)

OUT_BASE = "/content/splits"
IMAGES_DIR = "/content/dataset/images"

if os.path.exists(OUT_BASE):
    shutil.rmtree(OUT_BASE)

os.makedirs(OUT_BASE, exist_ok=True)

print(split_df.head())
print("\nSplit CSV rows:", len(split_df))

copied = 0
missing = 0

for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc="Copying split images"):
    filename = row["filename"]
    split = row["split"]
    label = row["label"]

    src = os.path.join(IMAGES_DIR, filename)
    dst_dir = os.path.join(OUT_BASE, split, label)
    dst = os.path.join(dst_dir, filename)

    os.makedirs(dst_dir, exist_ok=True)

    if os.path.exists(src):
        shutil.copy(src, dst)
        copied += 1
    else:
        missing += 1

print("Copied:", copied)
print("Missing:", missing)

   filename  split label
0  2150.jpg  train  real
1  3838.jpg  train  real
2  2018.jpg  train  real
3   342.jpg  train  real
4  1294.jpg  train  real

Split CSV rows: 8989


Copying split images:   0%|          | 0/8989 [00:00<?, ?it/s]

Copied: 8989
Missing: 0


In [5]:
import os
import pandas as pd

summary = []

for split_name in ["train", "val", "test"]:
    real_dir = f"/content/splits/{split_name}/real"
    fake_dir = f"/content/splits/{split_name}/fake"

    real_count = len(os.listdir(real_dir))
    fake_count = len(os.listdir(fake_dir))

    summary.append({
        "split": split_name,
        "real": real_count,
        "fake": fake_count,
        "total": real_count + fake_count
    })

summary_df = pd.DataFrame(summary)
print(summary_df)

   split  real  fake  total
0  train  2400  4791   7191
1    val   300   598    898
2   test   300   600    900


In [6]:
# =========================
# Create CNNDetection-compatible test folder
# ImageFolder uses alphabetical order:
# 0_real -> label 0
# 1_fake -> label 1
# =========================

import os, shutil

src_base = "/content/splits/test"
dst_base = "/content/splits/test_cnndetection"

if os.path.exists(dst_base):
    shutil.rmtree(dst_base)

os.makedirs(f"{dst_base}/0_real", exist_ok=True)
os.makedirs(f"{dst_base}/1_fake", exist_ok=True)

for fname in os.listdir(f"{src_base}/real"):
    shutil.copy(
        os.path.join(src_base, "real", fname),
        os.path.join(dst_base, "0_real", fname)
    )

for fname in os.listdir(f"{src_base}/fake"):
    shutil.copy(
        os.path.join(src_base, "fake", fname),
        os.path.join(dst_base, "1_fake", fname)
    )

print("Original test real:", len(os.listdir(f"{src_base}/real")))
print("Original test fake:", len(os.listdir(f"{src_base}/fake")))
print("CNNDetection 0_real:", len(os.listdir(f"{dst_base}/0_real")))
print("CNNDetection 1_fake:", len(os.listdir(f"{dst_base}/1_fake")))

Original test real: 300
Original test fake: 600
CNNDetection 0_real: 300
CNNDetection 1_fake: 600


## Clone Project

In [6]:
!git clone https://github.com/PeterWang512/CNNDetection

Cloning into 'CNNDetection'...
remote: Enumerating objects: 645, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 645 (delta 56), reused 50 (delta 50), pack-reused 568 (from 1)
Receiving objects: 100% (645/645), 6.37 MiB | 32.43 MiB/s, done.
Resolving deltas: 100% (106/106), done.


In [7]:
!pip install -r /content/CNNDetection/requirements.txt

In [8]:
!wget https://www.dropbox.com/s/2g2jagq2jn1fd0i/blur_jpg_prob0.5.pth?dl=0 -O /content/CNNDetection/weights/blur_jpg_prob0.5.pth
!wget https://www.dropbox.com/s/h7tkpcgiwuftb6g/blur_jpg_prob0.1.pth?dl=0 -O /content/CNNDetection/weights/blur_jpg_prob0.1.pth

--2026-05-19 19:29:06--  https://www.dropbox.com/s/2g2jagq2jn1fd0i/blur_jpg_prob0.5.pth?dl=0
Resolving www.dropbox.com (www.dropbox.com)... 162.125.1.18, 2620:100:6016:18::a27d:112
Connecting to www.dropbox.com (www.dropbox.com)|162.125.1.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/8wi9udqvub5mysawxzu31/blur_jpg_prob0.5.pth?rlkey=rnwce5afkayii29wlzllm09hf&dl=0 [following]
--2026-05-19 19:29:06--  https://www.dropbox.com/scl/fi/8wi9udqvub5mysawxzu31/blur_jpg_prob0.5.pth?rlkey=rnwce5afkayii29wlzllm09hf&dl=0
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://ucdde028cfbb7ee96727aeff0d26.dl.dropboxusercontent.com/cd/0/inline/DAyp1jQXWl-bmK4DCXwPFMcTwaYoQylrFgHPmgSNaVAad_6P8JJxuxGdTJCH9V9jl5DAeTScMHR-tmif9OzExnIA290M8iNpykcR7SI0w471BBwyzMFrQySr6kruBPZe0VVdSK-LUHvB5TkIx4z_37bd/file# [following]
--2026-05-19 19:29:07--  https://ucdde028cfbb7ee96727aeff0d26.

## Run Zero Shot on Test Split

In [ ]:
!python /content/CNNDetection/demo_dir.py \
  -d /content/splits/test_cnndetection \
  -m /content/CNNDetection/weights/blur_jpg_prob0.5.pth \
  -c 224 \
  -b 32

Cropping to [224]
Loading [1] datasets
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
  0% 0/29 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worke

In [33]:
TEST_DIR = "/content/splits/test"

test_ds = FolderDataset(TEST_DIR, eval_tfms)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

ckpt = torch.load(
    "/content/CNNDetection/weights/blur_jpg_prob0.5.pth",
    map_location=device
)

model.load_state_dict(ckpt["model"])
model.to(device)
model.eval()

test_acc, test_ap, test_auc, test_f1, test_cm = evaluate(test_loader)

tn, fp, fn, tp = test_cm.ravel()

real_recall = tn / (tn + fp)
fake_recall = tp / (tp + fn)

print("Zero-shot CNNDetection Results")
print(f"Accuracy: {test_acc*100:.2f}")
print(f"AP: {test_ap*100:.2f}")
print(f"AUC: {test_auc*100:.2f}")
print(f"F1: {test_f1*100:.2f}")
print("Confusion Matrix:")
print(test_cm)
print(f"Real Recall: {real_recall*100:.2f}")
print(f"Fake Recall: {fake_recall*100:.2f}")

Zero-shot CNNDetection Results
Accuracy: 33.78
AP: 71.53
AUC: 54.60
F1: 1.32
Confusion Matrix:
[[300   0]
 [596   4]]
Real Recall: 100.00
Fake Recall: 0.67


## Fine Tune CNN Detection (Legacy)

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    roc_auc_score,
    f1_score,
    confusion_matrix
)

# =========================
# Import CNNDetection ResNet
# =========================

sys.path.insert(0, "/content/CNNDetection")

from networks.resnet import resnet50

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# Dataset
# =========================

class FolderDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform

        real_dir = os.path.join(root_dir, "real")
        fake_dir = os.path.join(root_dir, "fake")

        for fname in os.listdir(real_dir):
            self.samples.append(
                (os.path.join(real_dir, fname), 0)
            )

        for fname in os.listdir(fake_dir):
            self.samples.append(
                (os.path.join(fake_dir, fname), 1)
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# =========================
# Transforms
# =========================

train_tfms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

eval_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

# =========================
# DataLoaders
# =========================

train_ds = FolderDataset("/content/splits/train", train_tfms)
val_ds = FolderDataset("/content/splits/val", eval_tfms)
test_ds = FolderDataset("/content/splits/test", eval_tfms)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

print("Train:", len(train_ds))
print("Val:", len(val_ds))
print("Test:", len(test_ds))

# =========================
# Load pretrained CNNDetection
# =========================

model = resnet50(num_classes=1)

ckpt = torch.load(
    "/content/CNNDetection/weights/blur_jpg_prob0.5.pth",
    map_location="cpu"
)

model.load_state_dict(ckpt["model"])

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False

# Train only final layer first
for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

# =========================
# Loss / Optimizer
# =========================

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.fc.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# =========================
# Evaluation
# =========================

def evaluate(loader):
    model.eval()

    y_true = []
    y_score = []

    with torch.no_grad():
        for images, labels in loader:

            images = images.to(device)
            labels = labels.float().to(device)

            logits = model(images).squeeze(1)

            probs = torch.sigmoid(logits)

            y_true.extend(labels.cpu().numpy())
            y_score.extend(probs.cpu().numpy())

    y_true = np.array(y_true)
    y_score = np.array(y_score)

    y_pred = (y_score >= 0.5).astype(int)

    acc = accuracy_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    return acc, ap, auc, f1, cm

# =========================
# Training
# =========================

best_auc = 0

for epoch in range(3):

    model.train()

    running_loss = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/3"):

        images = images.to(device)
        labels = labels.float().to(device)

        optimizer.zero_grad()

        logits = model(images).squeeze(1)

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    val_acc, val_ap, val_auc, val_f1, val_cm = evaluate(val_loader)

    print(f"\nEpoch {epoch+1}")

    print(f"Loss: {running_loss / len(train_loader):.4f}")

    print(f"Val Accuracy: {val_acc*100:.2f}")
    print(f"Val AP: {val_ap*100:.2f}")
    print(f"Val AUC: {val_auc*100:.2f}")
    print(f"Val F1: {val_f1*100:.2f}")

    print("Val Confusion Matrix:")
    print(val_cm)

    if val_auc > best_auc:

        best_auc = val_auc

        torch.save(
            model.state_dict(),
            "/content/best_cnndetection_finetuned.pth"
        )

        print("Saved best model")

Device: cuda
Train: 7191
Val: 898
Test: 900


Epoch 1/3:   0%|          | 0/225 [00:00<?, ?it/s]


Epoch 1
Loss: 3.4646
Val Accuracy: 54.23
Val AP: 70.61
Val AUC: 54.37
Val F1: 62.40
Val Confusion Matrix:
[[146 154]
 [257 341]]
Saved best model


Epoch 2/3:   0%|          | 0/225 [00:00<?, ?it/s]


Epoch 2
Loss: 1.2134
Val Accuracy: 55.35
Val AP: 68.01
Val AUC: 50.51
Val F1: 67.21
Val Confusion Matrix:
[[ 86 214]
 [187 411]]


Epoch 3/3:   0%|          | 0/225 [00:00<?, ?it/s]


Epoch 3
Loss: 0.8956
Val Accuracy: 55.68
Val AP: 67.38
Val AUC: 49.88
Val F1: 67.22
Val Confusion Matrix:
[[ 92 208]
 [190 408]]


### Test

In [ ]:
model.load_state_dict(
    torch.load(
        "/content/best_cnndetection_finetuned.pth",
        map_location=device
    )
)

test_acc, test_ap, test_auc, test_f1, test_cm = evaluate(test_loader)

print("Final Test Results - Fine-tuned CNNDetection")
print(f"Accuracy: {test_acc*100:.2f}")
print(f"AP: {test_ap*100:.2f}")
print(f"AUC: {test_auc*100:.2f}")
print(f"F1: {test_f1*100:.2f}")
print("Confusion Matrix:")
print(test_cm)

tn, fp, fn, tp = test_cm.ravel()

real_acc = tn / (tn + fp)
fake_acc = tp / (tp + fn)

print(f"Real Accuracy: {real_acc*100:.2f}")
print(f"Fake Accuracy: {fake_acc*100:.2f}")

Final Test Results - Fine-tuned CNNDetection
Accuracy: 55.56
AP: 71.78
AUC: 56.82
F1: 63.70
Confusion Matrix:
[[149 151]
 [249 351]]
Real Accuracy: 49.67
Fake Accuracy: 58.50


## Fine Tune Agreed Hyper Parameters

#### Load Without Train

In [9]:
import torch.nn as nn
import torch.utils.model_zoo as model_zoo


__all__ = ['ResNet', 'resnet18', 'resnet34', 'resnet50', 'resnet101',
           'resnet152']


model_urls = {
    'resnet18': 'https://download.pytorch.org/models/resnet18-5c106cde.pth',
    'resnet34': 'https://download.pytorch.org/models/resnet34-333f7ec4.pth',
    'resnet50': 'https://download.pytorch.org/models/resnet50-19c8e357.pth',
    'resnet101': 'https://download.pytorch.org/models/resnet101-5d3b4d8f.pth',
    'resnet152': 'https://download.pytorch.org/models/resnet152-b121ed2d.pth',
}


def conv3x3(in_planes, out_planes, stride=1):
    """3x3 convolution with padding"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False)


def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = conv1x1(inplanes, planes)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = conv3x3(planes, planes, stride)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = conv1x1(planes, planes * self.expansion)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out


class ResNet(nn.Module):

    def __init__(self, block, layers, num_classes=1000, zero_init_residual=False):
        super(ResNet, self).__init__()
        self.inplanes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3,
                               bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        # Zero-initialize the last BN in each residual branch,
        # so that the residual branch starts with zeros, and each residual block behaves like an identity.
        # This improves the model by 0.2~0.3% according to https://arxiv.org/abs/1706.02677
        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, Bottleneck):
                    nn.init.constant_(m.bn3.weight, 0)
                elif isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x


def resnet18(pretrained=False, **kwargs):
    """Constructs a ResNet-18 model.
    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = ResNet(BasicBlock, [2, 2, 2, 2], **kwargs)
    if pretrained:
        model.load_state_dict(model_zoo.load_url(model_urls['resnet18']))
    return model


def resnet34(pretrained=False, **kwargs):
    """Constructs a ResNet-34 model.
    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = ResNet(BasicBlock, [3, 4, 6, 3], **kwargs)
    if pretrained:
        model.load_state_dict(model_zoo.load_url(model_urls['resnet34']))
    return model


def resnet50(pretrained=False, **kwargs):
    """Constructs a ResNet-50 model.
    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = ResNet(Bottleneck, [3, 4, 6, 3], **kwargs)
    if pretrained:
        model.load_state_dict(model_zoo.load_url(model_urls['resnet50']))
    return model


def resnet101(pretrained=False, **kwargs):
    """Constructs a ResNet-101 model.
    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = ResNet(Bottleneck, [3, 4, 23, 3], **kwargs)
    if pretrained:
        model.load_state_dict(model_zoo.load_url(model_urls['resnet101']))
    return model


def resnet152(pretrained=False, **kwargs):
    """Constructs a ResNet-152 model.
    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = ResNet(Bottleneck, [3, 8, 36, 3], **kwargs)
    if pretrained:
        model.load_state_dict(model_zoo.load_url(model_urls['resnet152']))
    return model


In [10]:
import os, sys, torch
import torch.nn as nn
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score, average_precision_score, roc_auc_score, f1_score, confusion_matrix

sys.path.insert(0, "/content/CNNDetection")
from networks.resnet import resnet50

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

class FolderDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform

        for fname in os.listdir(os.path.join(root_dir, "real")):
            self.samples.append((os.path.join(root_dir, "real", fname), 0))

        for fname in os.listdir(os.path.join(root_dir, "fake")):
            self.samples.append((os.path.join(root_dir, "fake", fname), 1))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

train_tfms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

eval_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_ds = FolderDataset("/content/splits/train", train_tfms)
val_ds = FolderDataset("/content/splits/val", eval_tfms)
test_ds = FolderDataset("/content/splits/test", eval_tfms)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)

print("Train:", len(train_ds))
print("Val:", len(val_ds))
print("Test:", len(test_ds))

model = resnet50(num_classes=1)

ckpt = torch.load("/content/CNNDetection/weights/blur_jpg_prob0.5.pth", map_location="cpu")
model.load_state_dict(ckpt["model"])

# Freeze all
for param in model.parameters():
    param.requires_grad = False

# Unfreeze layer4 + fc
for param in model.layer4.parameters():
    param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Trainable params: {trainable_params:,}")
print(f"Total params: {total_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.2f}%")

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

def evaluate(loader):
    model.eval()
    y_true, y_score = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().to(device)

            logits = model(images).squeeze(1)
            probs = torch.sigmoid(logits)

            y_true.extend(labels.cpu().numpy())
            y_score.extend(probs.cpu().numpy())

    y_true = np.array(y_true)
    y_score = np.array(y_score)
    y_pred = (y_score >= 0.5).astype(int)

    acc = accuracy_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    return acc, ap, auc, f1, cm

Device: cuda
Train: 7191
Val: 898
Test: 900
Trainable params: 14,966,785
Total params: 23,510,081
Trainable %: 63.66%


#### Load with Train

In [ ]:
import os, sys, torch
import torch.nn as nn
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score, average_precision_score, roc_auc_score, f1_score, confusion_matrix

sys.path.insert(0, "/content/CNNDetection")
from networks.resnet import resnet50

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

class FolderDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform

        for fname in os.listdir(os.path.join(root_dir, "real")):
            self.samples.append((os.path.join(root_dir, "real", fname), 0))

        for fname in os.listdir(os.path.join(root_dir, "fake")):
            self.samples.append((os.path.join(root_dir, "fake", fname), 1))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

train_tfms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

eval_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_ds = FolderDataset("/content/splits/train", train_tfms)
val_ds = FolderDataset("/content/splits/val", eval_tfms)
test_ds = FolderDataset("/content/splits/test", eval_tfms)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)

print("Train:", len(train_ds))
print("Val:", len(val_ds))
print("Test:", len(test_ds))

model = resnet50(num_classes=1)

ckpt = torch.load("/content/CNNDetection/weights/blur_jpg_prob0.5.pth", map_location="cpu")
model.load_state_dict(ckpt["model"])

# Freeze all
for param in model.parameters():
    param.requires_grad = False

# Unfreeze layer4 + fc
for param in model.layer4.parameters():
    param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Trainable params: {trainable_params:,}")
print(f"Total params: {total_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.2f}%")

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

def evaluate(loader):
    model.eval()
    y_true, y_score = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().to(device)

            logits = model(images).squeeze(1)
            probs = torch.sigmoid(logits)

            y_true.extend(labels.cpu().numpy())
            y_score.extend(probs.cpu().numpy())

    y_true = np.array(y_true)
    y_score = np.array(y_score)
    y_pred = (y_score >= 0.5).astype(int)

    acc = accuracy_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    return acc, ap, auc, f1, cm

best_auc = 0
patience = 3
epochs_no_improve = 0
max_epochs = 10

for epoch in range(max_epochs):
    model.train()
    running_loss = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{max_epochs}"):
        images = images.to(device)
        labels = labels.float().to(device)

        optimizer.zero_grad()

        logits = model(images).squeeze(1)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    val_acc, val_ap, val_auc, val_f1, val_cm = evaluate(val_loader)
    scheduler.step(val_auc)

    current_lr = optimizer.param_groups[0]["lr"]

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {running_loss / len(train_loader):.4f}")
    print(f"LR: {current_lr:.8f}")
    print(f"Val Accuracy: {val_acc*100:.2f}")
    print(f"Val AP: {val_ap*100:.2f}")
    print(f"Val AUC: {val_auc*100:.2f}")
    print(f"Val F1: {val_f1*100:.2f}")
    print("Val Confusion Matrix:")
    print(val_cm)

    if val_auc > best_auc:
        best_auc = val_auc
        epochs_no_improve = 0
        torch.save(model.state_dict(), "/content/best_cnndetection_layer4_fc.pth")
        print("Saved best model ✅")
    else:
        epochs_no_improve += 1
        print(f"No improvement: {epochs_no_improve}/{patience}")

    if epochs_no_improve >= patience:
        print("Early stopping triggered.")
        break

Device: cuda
Train: 7191
Val: 898
Test: 900
Trainable params: 14,966,785
Total params: 23,510,081
Trainable %: 63.66%


Epoch 1/10:   0%|          | 0/899 [00:00<?, ?it/s]


Epoch 1
Loss: 0.8940
LR: 0.00010000
Val Accuracy: 78.29
Val AP: 90.66
Val AUC: 83.91
Val F1: 84.56
Val Confusion Matrix:
[[169 131]
 [ 64 534]]
Saved best model ✅


Epoch 2/10:   0%|          | 0/899 [00:00<?, ?it/s]


Epoch 2
Loss: 0.4990
LR: 0.00010000
Val Accuracy: 81.07
Val AP: 92.92
Val AUC: 87.22
Val F1: 86.64
Val Confusion Matrix:
[[177 123]
 [ 47 551]]
Saved best model ✅


Epoch 3/10:   0%|          | 0/899 [00:00<?, ?it/s]

... (13 outputs left out)

Epoch 9/10:   0%|          | 0/899 [00:00<?, ?it/s]


Epoch 9
Loss: 0.3321
LR: 0.00010000
Val Accuracy: 86.86
Val AP: 96.35
Val AUC: 93.23
Val F1: 90.31
Val Confusion Matrix:
[[230  70]
 [ 48 550]]
Saved best model ✅


Epoch 10/10:   0%|          | 0/899 [00:00<?, ?it/s]


Epoch 10
Loss: 0.3168
LR: 0.00010000
Val Accuracy: 84.08
Val AP: 96.08
Val AUC: 92.79
Val F1: 88.75
Val Confusion Matrix:
[[191 109]
 [ 34 564]]
No improvement: 1/3


### Test

In [ ]:
model.load_state_dict(torch.load("/content/best_cnndetection_layer4_fc.pth", map_location=device))

test_acc, test_ap, test_auc, test_f1, test_cm = evaluate(test_loader)

tn, fp, fn, tp = test_cm.ravel()

real_acc = tn / (tn + fp)
fake_acc = tp / (tp + fn)

print("Final Test Results - CNNDetection Fine-tuned layer4 + fc")
print(f"Accuracy: {test_acc*100:.2f}")
print(f"AP: {test_ap*100:.2f}")
print(f"AUC: {test_auc*100:.2f}")
print(f"F1: {test_f1*100:.2f}")
print("Confusion Matrix:")
print(test_cm)
print(f"Real Accuracy: {real_acc*100:.2f}")
print(f"Fake Accuracy: {fake_acc*100:.2f}")

Final Test Results - CNNDetection Fine-tuned layer4 + fc
Accuracy: 87.00
AP: 96.81
AUC: 93.80
F1: 90.50
Confusion Matrix:
[[226  74]
 [ 43 557]]
Real Accuracy: 75.33
Fake Accuracy: 92.83


### Per-Model Test

In [12]:
!cp /content/dataset/metadata.csv  /content/splits

## Cross Generalization on Out Of Distribution Dataset

### Extract Data Set

In [16]:
import os, shutil
from pathlib import Path

src_root = Path("/content/drive/MyDrive/Image Processing Project/OOD Dataset/openfake_ood_test_256/ood_test")
dst_root = Path("/content/ood_test_450_real")

# create folders
(dst_root / "real").mkdir(parents=True, exist_ok=True)
(dst_root / "fake").mkdir(parents=True, exist_ok=True)

# take first 450 real images
real_imgs = sorted(list((src_root / "real").glob("*")))[:450]

for img in real_imgs:
    shutil.copy(img, dst_root / "real" / img.name)

# copy all fake images
fake_imgs = sorted(list((src_root / "fake").glob("*")))

for img in fake_imgs:
    shutil.copy(img, dst_root / "fake" / img.name)

print("Real copied:", len(list((dst_root / "real").glob("*"))))
print("Fake copied:", len(list((dst_root / "fake").glob("*"))))
print("New test path:", dst_root)

Real copied: 450
Fake copied: 900
New test path: /content/ood_test_450_real


### Fine Tuned Test

In [18]:
TEST_DIR = "/content/ood_test_450_real"

test_ds = FolderDataset(TEST_DIR, eval_tfms)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)

print("Test:", len(test_ds))

model.load_state_dict(torch.load(
    "/content/best_cnndetection_layer4_fc.pth",
    map_location=device
))

model.to(device)
model.eval()

test_acc, test_ap, test_auc, test_f1, test_cm = evaluate(test_loader)

tn, fp, fn, tp = test_cm.ravel()

real_acc = tn / (tn + fp)
fake_acc = tp / (tp + fn)

print("OOD Test Results - CNNDetection Fine-tuned layer4 + fc")
print(f"Accuracy: {test_acc*100:.2f}")
print(f"AP: {test_ap*100:.2f}")
print(f"AUC: {test_auc*100:.2f}")
print(f"F1: {test_f1*100:.2f}")
print("Confusion Matrix:")
print(test_cm)
print(f"Real Accuracy: {real_acc*100:.2f}")
print(f"Fake Accuracy: {fake_acc*100:.2f}")

Test: 1350
OOD Test Results - CNNDetection Fine-tuned layer4 + fc
Accuracy: 60.52
AP: 73.30
AUC: 57.90
F1: 71.51
Confusion Matrix:
[[148 302]
 [231 669]]
Real Accuracy: 32.89
Fake Accuracy: 74.33


### Zero Shot Test

In [19]:
TEST_DIR = "/content/ood_test_450_real"

test_ds = FolderDataset(TEST_DIR, eval_tfms)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

ckpt = torch.load(
    "/content/CNNDetection/weights/blur_jpg_prob0.5.pth",
    map_location=device
)

model.load_state_dict(ckpt["model"])
model.to(device)
model.eval()

test_acc, test_ap, test_auc, test_f1, test_cm = evaluate(test_loader)

tn, fp, fn, tp = test_cm.ravel()

real_recall = tn / (tn + fp)
fake_recall = tp / (tp + fn)

print("Zero-shot CNNDetection Results")
print(f"Accuracy: {test_acc*100:.2f}")
print(f"AP: {test_ap*100:.2f}")
print(f"AUC: {test_auc*100:.2f}")
print(f"F1: {test_f1*100:.2f}")
print("Confusion Matrix:")
print(test_cm)
print(f"Real Recall: {real_recall*100:.2f}")
print(f"Fake Recall: {fake_recall*100:.2f}")

Zero-shot CNNDetection Results
Accuracy: 33.04
AP: 52.99
AUC: 27.54
F1: 0.00
Confusion Matrix:
[[446   4]
 [900   0]]
Real Recall: 99.11
Fake Recall: 0.00


## Fine Tune ResNet-50 (Skipped)

In [ ]:
import os, torch
import torch.nn as nn
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, average_precision_score, roc_auc_score, f1_score, confusion_matrix

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

class FolderDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform

        for fname in os.listdir(os.path.join(root_dir, "real")):
            self.samples.append((os.path.join(root_dir, "real", fname), 0))

        for fname in os.listdir(os.path.join(root_dir, "fake")):
            self.samples.append((os.path.join(root_dir, "fake", fname), 1))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

train_tfms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

eval_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_ds = FolderDataset("/content/splits/train", train_tfms)
val_ds = FolderDataset("/content/splits/val", eval_tfms)
test_ds = FolderDataset("/content/splits/test", eval_tfms)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)

print("Train:", len(train_ds))
print("Val:", len(val_ds))
print("Test:", len(test_ds))

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace classifier head
model.fc = nn.Linear(model.fc.in_features, 1)

# Unfreeze layer4 + fc
for param in model.layer4.parameters():
    param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Trainable params: {trainable_params:,}")
print(f"Total params: {total_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.2f}%")

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

def evaluate(loader):
    model.eval()
    y_true, y_score = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().to(device)

            logits = model(images).squeeze(1)
            probs = torch.sigmoid(logits)

            y_true.extend(labels.cpu().numpy())
            y_score.extend(probs.cpu().numpy())

    y_true = np.array(y_true)
    y_score = np.array(y_score)
    y_pred = (y_score >= 0.5).astype(int)

    acc = accuracy_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    return acc, ap, auc, f1, cm

best_auc = 0
patience = 3
epochs_no_improve = 0
max_epochs = 10

for epoch in range(max_epochs):
    model.train()
    running_loss = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{max_epochs}"):
        images = images.to(device)
        labels = labels.float().to(device)

        optimizer.zero_grad()
        logits = model(images).squeeze(1)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    val_acc, val_ap, val_auc, val_f1, val_cm = evaluate(val_loader)
    scheduler.step(val_auc)

    current_lr = optimizer.param_groups[0]["lr"]

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {running_loss / len(train_loader):.4f}")
    print(f"LR: {current_lr:.8f}")
    print(f"Val Accuracy: {val_acc*100:.2f}")
    print(f"Val AP: {val_ap*100:.2f}")
    print(f"Val AUC: {val_auc*100:.2f}")
    print(f"Val F1: {val_f1*100:.2f}")
    print("Val Confusion Matrix:")
    print(val_cm)

    if val_auc > best_auc:
        best_auc = val_auc
        epochs_no_improve = 0
        torch.save(model.state_dict(), "/content/best_imagenet_resnet50_layer4_fc.pth")
        print("Saved best model ✅")
    else:
        epochs_no_improve += 1
        print(f"No improvement: {epochs_no_improve}/{patience}")

    if epochs_no_improve >= patience:
        print("Early stopping triggered.")
        break

Device: cuda
Train: 7191
Val: 898
Test: 900
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 133MB/s]


Trainable params: 14,966,785
Total params: 23,510,081
Trainable %: 63.66%


Epoch 1/10:   0%|          | 0/899 [00:00<?, ?it/s]


Epoch 1
Loss: 0.4029
LR: 0.00010000
Val Accuracy: 87.86
Val AP: 97.69
Val AUC: 95.52
Val F1: 91.19
Val Confusion Matrix:
[[225  75]
 [ 34 564]]
Saved best model ✅


Epoch 2/10:   0%|          | 0/899 [00:00<?, ?it/s]

... (6 outputs left out)

Epoch 5/10:   0%|          | 0/899 [00:00<?, ?it/s]


Epoch 5
Loss: 0.1609
LR: 0.00010000
Val Accuracy: 91.31
Val AP: 98.37
Val AUC: 96.99
Val F1: 93.37
Val Confusion Matrix:
[[271  29]
 [ 49 549]]
No improvement: 2/3


Epoch 6/10:   0%|          | 0/899 [00:00<?, ?it/s]


Epoch 6
Loss: 0.1377
LR: 0.00010000
Val Accuracy: 90.31
Val AP: 98.29
Val AUC: 96.62
Val F1: 92.76
Val Confusion Matrix:
[[254  46]
 [ 41 557]]
No improvement: 3/3
Early stopping triggered.


### Test

In [ ]:
model.load_state_dict(torch.load("/content/best_imagenet_resnet50_layer4_fc.pth", map_location=device))

test_acc, test_ap, test_auc, test_f1, test_cm = evaluate(test_loader)

tn, fp, fn, tp = test_cm.ravel()

real_acc = tn / (tn + fp)
fake_acc = tp / (tp + fn)

print("Final Test Results - ImageNet ResNet-50 Fine-tuned layer4 + fc")
print(f"Accuracy: {test_acc*100:.2f}")
print(f"AP: {test_ap*100:.2f}")
print(f"AUC: {test_auc*100:.2f}")
print(f"F1: {test_f1*100:.2f}")
print("Confusion Matrix:")
print(test_cm)
print(f"Real Accuracy: {real_acc*100:.2f}")
print(f"Fake Accuracy: {fake_acc*100:.2f}")

Final Test Results - ImageNet ResNet-50 Fine-tuned layer4 + fc
Accuracy: 89.44
AP: 97.97
AUC: 96.22
F1: 92.41
Confusion Matrix:
[[227  73]
 [ 22 578]]
Real Accuracy: 75.67
Fake Accuracy: 96.33


## Cross Generalization (Skipped)

In [ ]:
import pandas as pd
import os
import shutil
from sklearn.model_selection import train_test_split

# =========================
# Load metadata
# =========================

df = pd.read_csv("/content/dataset/metadata.csv")

df["image_path"] = df["image_path"].str.replace(
    "/content/openfake_subset/images",
    "/content/dataset/images",
    regex=False
)

# =========================
# Separate datasets
# =========================

real_df = df[df["label"] == "real"].copy()

mid_df = df[df["label"] == "midjourney"].copy()

sd_df = df[df["label"] == "stable_diffusion"].copy()

print("Real:", len(real_df))
print("Midjourney:", len(mid_df))
print("Stable Diffusion:", len(sd_df))

# =========================
# Train/Val from Midjourney
# =========================

train_real, temp_real = train_test_split(
    real_df,
    test_size=0.2,
    random_state=42
)

val_real, test_real = train_test_split(
    temp_real,
    test_size=0.5,
    random_state=42
)

train_mid, val_mid = train_test_split(
    mid_df,
    test_size=0.1,
    random_state=42
)

# =========================
# Test ONLY on Stable Diffusion
# =========================

test_sd = sd_df.sample(n=len(test_real), random_state=42)

# =========================
# Build folder structure
# =========================

BASE = "/content/cross_gen"

if os.path.exists(BASE):
    shutil.rmtree(BASE)

splits = [
    ("train", train_real, train_mid),
    ("val", val_real, val_mid),
]

for split_name, real_split, fake_split in splits:

    os.makedirs(f"{BASE}/{split_name}/real", exist_ok=True)
    os.makedirs(f"{BASE}/{split_name}/fake", exist_ok=True)

    for _, row in real_split.iterrows():
        shutil.copy(
            row["image_path"],
            f"{BASE}/{split_name}/real/{os.path.basename(row['image_path'])}"
        )

    for _, row in fake_split.iterrows():
        shutil.copy(
            row["image_path"],
            f"{BASE}/{split_name}/fake/{os.path.basename(row['image_path'])}"
        )

# =========================
# Test set:
# real + Stable Diffusion
# =========================

os.makedirs(f"{BASE}/test/real", exist_ok=True)
os.makedirs(f"{BASE}/test/fake", exist_ok=True)

for _, row in test_real.iterrows():
    shutil.copy(
        row["image_path"],
        f"{BASE}/test/real/{os.path.basename(row['image_path'])}"
    )

for _, row in test_sd.iterrows():
    shutil.copy(
        row["image_path"],
        f"{BASE}/test/fake/{os.path.basename(row['image_path'])}"
    )

print("\nDone creating cross-generalization dataset.")

for split_name in ["train", "val", "test"]:
    r = len(os.listdir(f"{BASE}/{split_name}/real"))
    f = len(os.listdir(f"{BASE}/{split_name}/fake"))

    print(f"{split_name}: real={r}, fake={f}")

Real: 3000
Midjourney: 2989
Stable Diffusion: 3000

Done creating cross-generalization dataset.
train: real=2400, fake=2690
val: real=300, fake=299
test: real=300, fake=300


In [ ]:
import os, torch
import torch.nn as nn
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, average_precision_score, roc_auc_score, f1_score, confusion_matrix

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

class FolderDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform

        for fname in os.listdir(os.path.join(root_dir, "real")):
            self.samples.append((os.path.join(root_dir, "real", fname), 0))

        for fname in os.listdir(os.path.join(root_dir, "fake")):
            self.samples.append((os.path.join(root_dir, "fake", fname), 1))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

train_tfms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

eval_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_ds = FolderDataset("/content/cross_gen/train", train_tfms)
val_ds = FolderDataset("/content/cross_gen/val", eval_tfms)
test_ds = FolderDataset("/content/cross_gen/test", eval_tfms)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)

print("Train:", len(train_ds))
print("Val:", len(val_ds))
print("Test:", len(test_ds))

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 1)

for param in model.layer4.parameters():
    param.requires_grad = True

for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Trainable params: {trainable_params:,}")
print(f"Total params: {total_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.2f}%")

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

def evaluate(loader):
    model.eval()
    y_true, y_score = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().to(device)

            logits = model(images).squeeze(1)
            probs = torch.sigmoid(logits)

            y_true.extend(labels.cpu().numpy())
            y_score.extend(probs.cpu().numpy())

    y_true = np.array(y_true)
    y_score = np.array(y_score)
    y_pred = (y_score >= 0.5).astype(int)

    acc = accuracy_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    return acc, ap, auc, f1, cm

best_auc = 0
patience = 3
epochs_no_improve = 0
max_epochs = 10

for epoch in range(max_epochs):
    model.train()
    running_loss = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{max_epochs}"):
        images = images.to(device)
        labels = labels.float().to(device)

        optimizer.zero_grad()
        logits = model(images).squeeze(1)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    val_acc, val_ap, val_auc, val_f1, val_cm = evaluate(val_loader)
    scheduler.step(val_auc)

    current_lr = optimizer.param_groups[0]["lr"]

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {running_loss / len(train_loader):.4f}")
    print(f"LR: {current_lr:.8f}")
    print(f"Val Accuracy: {val_acc*100:.2f}")
    print(f"Val AP: {val_ap*100:.2f}")
    print(f"Val AUC: {val_auc*100:.2f}")
    print(f"Val F1: {val_f1*100:.2f}")
    print("Val Confusion Matrix:")
    print(val_cm)

    if val_auc > best_auc:
        best_auc = val_auc
        epochs_no_improve = 0
        torch.save(model.state_dict(), "/content/best_crossgen_mj_to_sd_resnet50.pth")
        print("Saved best model ✅")
    else:
        epochs_no_improve += 1
        print(f"No improvement: {epochs_no_improve}/{patience}")

    if epochs_no_improve >= patience:
        print("Early stopping triggered.")
        break

Device: cuda
Train: 5090
Val: 599
Test: 600
Trainable params: 14,966,785
Total params: 23,510,081
Trainable %: 63.66%


Epoch 1/10:   0%|          | 0/637 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x79a029cd1760>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x79a029cd1760>
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    ^self._shutdown_workers()^
^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
        assert self._parent_pid == os.getpid(), 'can only test a child process'if w.is_alive():

              ^ ^  ^^ ^^^^^^^^^^^^^^^^^^^^^


Epoch 1
Loss: 0.2549
LR: 0.00010000
Val Accuracy: 92.65
Val AP: 98.34
Val AUC: 98.49
Val F1: 93.08
Val Confusion Matrix:
[[259  41]
 [  3 296]]
Saved best model ✅


Epoch 2/10:   0%|          | 0/637 [00:00<?, ?it/s]


Epoch 2
Loss: 0.1333
LR: 0.00010000
Val Accuracy: 93.82
Val AP: 98.85
Val AUC: 98.78
Val F1: 94.12
Val Confusion Matrix:
[[266  34]
 [  3 296]]
Saved best model ✅


... (4 outputs left out)

Epoch 5/10:   0%|          | 0/637 [00:00<?, ?it/s]


Epoch 5
Loss: 0.0629
LR: 0.00010000
Val Accuracy: 96.33
Val AP: 98.76
Val AUC: 98.97
Val F1: 96.38
Val Confusion Matrix:
[[284  16]
 [  6 293]]
No improvement: 2/3


Epoch 6/10:   0%|          | 0/637 [00:00<?, ?it/s]


Epoch 6
Loss: 0.0451
LR: 0.00010000
Val Accuracy: 94.82
Val AP: 99.25
Val AUC: 99.10
Val F1: 95.02
Val Confusion Matrix:
[[272  28]
 [  3 296]]
No improvement: 3/3
Early stopping triggered.


### Test

In [ ]:
model.load_state_dict(torch.load("/content/best_crossgen_mj_to_sd_resnet50.pth", map_location=device))

test_acc, test_ap, test_auc, test_f1, test_cm = evaluate(test_loader)

tn, fp, fn, tp = test_cm.ravel()

real_acc = tn / (tn + fp)
fake_acc = tp / (tp + fn)

print("Final Test Results - Cross-Gen ResNet-50: Train MJ → Test SD")
print(f"Accuracy: {test_acc*100:.2f}")
print(f"AP: {test_ap*100:.2f}")
print(f"AUC: {test_auc*100:.2f}")
print(f"F1: {test_f1*100:.2f}")
print("Confusion Matrix:")
print(test_cm)
print(f"Real Accuracy: {real_acc*100:.2f}")
print(f"Fake Accuracy: {fake_acc*100:.2f}")

Final Test Results - Cross-Gen ResNet-50: Train MJ → Test SD
Accuracy: 62.00
AP: 76.81
AUC: 75.48
F1: 44.39
Confusion Matrix:
[[281  19]
 [209  91]]
Real Accuracy: 93.67
Fake Accuracy: 30.33
